In [ ]:
import polars as pl
import requests
import json
import numpy as np
import time
from tqdm import tqdm
import datetime

from constant import data_path


# extract arxiv preprint from arxiv_raw

In [12]:
import re
from collections import defaultdict
from datetime import datetime

logfile = "log/arxiv_log.log"

TOTAL = 0

# 正規表現
date_re = re.compile(r"Downloading for (\d{8})")
count_re = re.compile(r"Downloaded (\d+) entries")

monthly_counts = defaultdict(int)

current_date = None

with open(logfile, encoding="utf-8") as f:
    for line in f:
        m_date = date_re.search(line)
        if m_date:
            # YYYYMMDD → datetime
            current_date = datetime.strptime(m_date.group(1), "%Y%m%d")
            continue

        m_count = count_re.search(line)
        if m_count and current_date is not None:
            month_key = current_date.strftime("%Y-%m")
            monthly_counts[month_key] += int(m_count.group(1))
            current_date = None  # 念のためリセット

# 表示
for month in sorted(monthly_counts):
    TOTAL += monthly_counts[month]
    print(month, monthly_counts[month])
TOTAL

2019-01 11722
2019-02 11326
2019-03 13086
2019-04 12931
2019-05 13782
2019-06 13313
2019-07 13220
2019-08 11980
2019-09 13786
2019-10 14643
2019-11 13532
2019-12 12839
2020-01 11944
2020-02 13231
2020-03 14160
2020-04 15074
2020-05 15164
2020-06 16447
2020-07 16248
2020-08 13772
2020-09 14799
2020-10 16706
2020-11 14968
2020-12 15708
2021-01 13178
2021-02 13818
2021-03 18050
2021-04 15072
2021-05 15208
2021-06 16220
2021-07 15017
2021-08 15145
2021-09 20379
2021-10 16459
2021-11 15231
2021-12 15485
2022-01 13530
2022-02 14024
2022-03 17202
2022-04 14520
2022-05 15857
2022-06 15476
2022-07 15269
2022-08 14585
2022-09 15597
2022-10 17653
2022-11 17174
2022-12 15032
2023-01 13703
2023-02 14852
2023-03 18320
2023-04 15364
2023-05 19747
2023-06 17711
2023-07 17080
2023-08 16890
2023-09 17891
2023-10 20366
2023-11 18887
2023-12 18307
2024-01 17620
2024-02 19514
2024-03 20800
2024-04 19298
2024-05 21051
2024-06 20689
2024-07 20659
2024-08 17782
2024-09 20414
2024-10 24014
2024-11 20216
2024-1

1161960

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json
from datetime import datetime

NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

FIELDS = [
    "abstract", "acm_class", "arxiv_id", "author_text", "categories", "comments",
    "created", "doi", "num_authors", "num_categories", "primary_cat", "title",
    "updated", "created_ym", "license",
]

def _text_or_none(elem, path, default=None):
    n = elem.find(path, NS)
    if n is None or n.text is None:
        return default
    return n.text.strip()

def arxiv_xml_to_records(xml_path: Path) -> list[dict]:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    records: list[dict] = []
    for entry in root.findall("atom:entry", NS):
        id_url = _text_or_none(entry, "atom:id")
        arxiv_id = id_url.rsplit("/", 1)[-1] if id_url else None

        title = _text_or_none(entry, "atom:title", "")
        title = " ".join(title.split()) if title else None

        abstract = _text_or_none(entry, "atom:summary", "")
        abstract = " ".join(abstract.split()) if abstract else None

        updated = _text_or_none(entry, "atom:updated")
        created = _text_or_none(entry, "atom:published")  # created相当
        license = _text_or_none(entry, "arxiv:license")

        created_ym = None
        if created:
            try:
                dt = datetime.fromisoformat(created.replace("Z", "+00:00"))
                created_ym = dt.strftime("%Y-%m")
            except Exception:
                pass

        doi = _text_or_none(entry, "arxiv:doi")
        comments = _text_or_none(entry, "arxiv:comment")
        acm_class = _text_or_none(entry, "arxiv:acm-class")

        cat_terms = [
            c.get("term")
            for c in entry.findall("atom:category", NS)
            if c.get("term")
        ]
        categories = " ".join(cat_terms) if cat_terms else None

        pc = entry.find("arxiv:primary_category", NS)
        primary_cat = pc.get("term") if pc is not None else None

        authors = []
        for a in entry.findall("atom:author", NS):
            nm = _text_or_none(a, "atom:name")
            if nm:
                authors.append(nm)
        author_text = "; ".join(authors) if authors else None

        rec = {
            "abstract": abstract,
            "acm_class": acm_class,
            "arxiv_id": arxiv_id,
            "author_text": author_text,
            "categories": categories,
            "comments": comments,
            "created": created,
            "doi": doi,
            "num_authors": len(authors),
            "num_categories": len(cat_terms),
            "primary_cat": primary_cat,
            "title": title,
            "updated": updated,
            "created_ym": created_ym,
            "license": license,
        }

        # 念のため、列順を固定したい場合（JSONLだと必須ではないが）
        rec = {k: rec.get(k) for k in FIELDS}
        records.append(rec)

    return records


def month_dir_to_single_jsonl(raw_month_dir: Path, out_jsonl_path: Path) -> int:
    """
    raw_month_dir (例: data/arxiv_raw/201901/) 配下の *.xml をすべて読み、
    1か月分を1本の JSONL にまとめて出力する。
    書き込みは「1 XML につき 1 回」。
    """
    out_jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    total = 0
    xml_files = sorted(raw_month_dir.glob("*.xml"))

    with out_jsonl_path.open("w", encoding="utf-8") as f:
        for xml_path in xml_files:
            records = arxiv_xml_to_records(xml_path)
            if not records:
                continue

            # 1 XML = 1回の write（行ごとのI/Oはしない）
            chunk = "\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n"
            f.write(chunk)
            total += len(records)

    return total

In [2]:
"""
raw_month_dir (例: data/arxiv_raw/201901/) 配下の *.xml をすべて読み、
1か月分を1本の JSONL にまとめて出力する。
書き込みは「1 XML につき 1 回」。
"""
raw_root = Path("data/arxiv_raw")
year = 2019
month = 1
ym = f"{year}{month:02d}"
raw_month_dir = raw_root / ym
total = 0
xml_file = sorted(raw_month_dir.glob("*.xml"))[0]

records = arxiv_xml_to_records(xml_file)
# 1 XML = 1回の write（行ごとのI/Oはしない）
chunk = "\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n"
total += len(records)


In [11]:
df = pl.DataFrame(records)
df.group_by(pl.col('license').is_null()).len()

license,len
bool,u32
true,185


In [ ]:
from pathlib import Path

raw_root = Path("data/arxiv_raw")
out_root = Path("data/arxiv_jsonl")
out_root.mkdir(parents=True, exist_ok=True)

years = range(2019, 2025)   # 必要に応じて調整
months = range(1, 13)

for year in years:
    for month in months:
        ym = f"{year}{month:02d}"
        month_dir = raw_root / ym

        if not month_dir.is_dir():
            continue

        out_path = out_root / f"arxiv_{ym}.jsonl"

        if out_path.exists():
            raise FileExistsError(f"Output already exists: {out_path}")

        n = month_dir_to_single_jsonl(month_dir, out_path)
        print(f"{ym}: {n} records -> {out_path}")

201901: 11537 records -> data/arxiv_jsonl/arxiv_201901.jsonl
201902: 11326 records -> data/arxiv_jsonl/arxiv_201902.jsonl
201903: 13086 records -> data/arxiv_jsonl/arxiv_201903.jsonl
201904: 12931 records -> data/arxiv_jsonl/arxiv_201904.jsonl
201905: 13782 records -> data/arxiv_jsonl/arxiv_201905.jsonl
201906: 12734 records -> data/arxiv_jsonl/arxiv_201906.jsonl
201907: 13220 records -> data/arxiv_jsonl/arxiv_201907.jsonl
201908: 11980 records -> data/arxiv_jsonl/arxiv_201908.jsonl
201909: 13786 records -> data/arxiv_jsonl/arxiv_201909.jsonl
201910: 14643 records -> data/arxiv_jsonl/arxiv_201910.jsonl
201911: 13532 records -> data/arxiv_jsonl/arxiv_201911.jsonl
201912: 12839 records -> data/arxiv_jsonl/arxiv_201912.jsonl
202001: 11944 records -> data/arxiv_jsonl/arxiv_202001.jsonl
202002: 13231 records -> data/arxiv_jsonl/arxiv_202002.jsonl


In [24]:
df = pl.scan_ndjson("data/arxiv_jsonl/arxiv_*.jsonl")
df.head().collect(engine="streaming"), df.count().collect(engine="streaming")

(shape: (5, 14)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ abstract  ┆ acm_class ┆ arxiv_id  ┆ author_te ┆ … ┆ primary_c ┆ title     ┆ updated   ┆ created_ │
 │ ---       ┆ ---       ┆ ---       ┆ xt        ┆   ┆ at        ┆ ---       ┆ ---       ┆ ym       │
 │ str       ┆ null      ┆ str       ┆ ---       ┆   ┆ ---       ┆ str       ┆ str       ┆ ---      │
 │           ┆           ┆           ┆ str       ┆   ┆ str       ┆           ┆           ┆ str      │
 ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ In        ┆ null      ┆ 1901.0006 ┆ Mahtab    ┆ … ┆ cs.CL     ┆ Improving ┆ 2019-01-0 ┆ 2019-01  │
 │ Natural   ┆           ┆ 6v1       ┆ Ahmed;    ┆   ┆           ┆ Tree-LSTM ┆ 1T00:15:4 ┆          │
 │ Language  ┆           ┆           ┆ Muhammad  ┆   ┆           ┆ with Tree ┆ 5Z        ┆          │
 │ Processin ┆           ┆           ┆ Rifayat…  ┆   ┆           ┆

In [26]:
df.unique("arxiv_id").count().collect(engine="streaming")

abstract,acm_class,arxiv_id,author_text,categories,comments,created,doi,num_authors,num_categories,primary_cat,title,updated,created_ym
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1153832,0,1153832,1153832,1153832,752705,1153832,398509,1153832,1153832,1153832,1153832,1153832,1153832


# arxivArchive

In [12]:
# process arxiv archive [Geiger, R. S. (2019).]

import zipfile
import polars as pl

for year in range(1993, 2018 + 1):
    zip_path = f"/home/ec2-user/preprint_journal/temp/arxiv_archive-master/processed_data/20190101/per_year/{year}.tsv.zip"

    with zipfile.ZipFile(zip_path) as z:
        name = z.namelist()[0]             # ZIP 内の最初のファイル名
        with z.open(name) as f:
            df = pl.read_csv(
                f, 
                separator="\t", 
                infer_schema=False,
            )
    df = df.with_columns(
        pl.col("arxiv_id").cast(pl.String).fill_null(""),
        pl.col("acm_class").cast(pl.List(pl.String)).fill_null([""]),
        pl.col("categories").str.split(",").cast(pl.List(pl.String)),
    )
    # print(df.head())
    # break
    df.write_ndjson(f"{base_path}/full_{year}.jsonl")

In [14]:
import logging
import re
from dateutil.parser import parse as parse_date

logging.basicConfig(
    filename='log/fetch_arxiv.log',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logging.info("Starting the script")

In [16]:
df.dtypes

[String,
 String,
 List(String),
 String,
 String,
 List(String),
 String,
 String,
 String,
 String,
 String,
 String,
 String,
 String,
 String]

# export all dois and curated dois

In [ ]:
# concat all jsonl files vertically
columns = [
    'abstract',
    'arxiv_id',
    'categories',
    'comments',
    'doi',
    'num_categories',
    'primary_cat',
    'title',
    'created'
]

df_before = pl.concat([
    pl
    .read_ndjson(f"{data_path}/arxiv/full_{year}.jsonl")
    .select(columns)
    for year in range(1993, 2019)
])
df_after = (
    pl.read_ndjson(f"data/arxiv_jsonl/arxiv_*.jsonl")
    .select(columns)
)

df_before = (
    df_before
    .with_columns(
        pl.col("num_categories").cast(pl.Int8)
    )
    .with_columns(
        pl.col("created").str.slice(0, 4).str.to_integer().cast(pl.Int16).alias("pub_year")
    )
)
df_after = (
    df_after
    .with_columns(
        pl.col("num_categories").cast(pl.Int8),
        pl.col("categories").str.split(" ").cast(pl.List(pl.String)),
        pl.col("created").str.slice(0, 4).str.to_integer().cast(pl.Int16).alias("pub_year")
    )
    .with_columns(
        pl.col("arxiv_id").str.split("v").list[0].alias("arxiv_id"),
        pl.col("arxiv_id").str.split("v").list[1].str.to_integer().alias("version"),
    )
    # # arxiv id recorded only once with later version, there is no duplicate entries for multiple versions
    # .filter(
    #     pl.col("version") == 1 
    # )
)


In [59]:
df_before.head(), df_before.count()

(shape: (5, 10)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ abstract  ┆ arxiv_id  ┆ categorie ┆ comments  ┆ … ┆ primary_c ┆ title     ┆ created   ┆ pub_year │
 │ ---       ┆ ---       ┆ s         ┆ ---       ┆   ┆ at        ┆ ---       ┆ ---       ┆ ---      │
 │ str       ┆ str       ┆ ---       ┆ str       ┆   ┆ ---       ┆ str       ┆ str       ┆ i16      │
 │           ┆           ┆ list[str] ┆           ┆   ┆ str       ┆           ┆           ┆          │
 ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ We study  ┆ cond-mat/ ┆ ["cond-ma ┆ 12 pages, ┆ … ┆ cond-mat  ┆ Effect of ┆ 1993-01-0 ┆ 1993     │
 │ the       ┆ 9301001   ┆ t"]       ┆ LaTeX,    ┆   ┆           ┆ Loops on  ┆ 1         ┆          │
 │ effects   ┆           ┆           ┆ HLRZ      ┆   ┆           ┆ the       ┆           ┆          │
 │ of        ┆           ┆           ┆ 107/92    ┆   ┆           ┆

In [60]:
df_after.head(), df_after.count()

(shape: (5, 11)
 ┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬─────────┐
 │ abstract   ┆ arxiv_id   ┆ categorie ┆ comments  ┆ … ┆ title     ┆ created   ┆ pub_year ┆ version │
 │ ---        ┆ ---        ┆ s         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---      ┆ ---     │
 │ str        ┆ str        ┆ ---       ┆ str       ┆   ┆ str       ┆ str       ┆ i16      ┆ i64     │
 │            ┆            ┆ list[str] ┆           ┆   ┆           ┆           ┆          ┆         │
 ╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═════════╡
 │ In Natural ┆ 1901.00066 ┆ ["cs.CL", ┆ 8 Pages,  ┆ … ┆ Improving ┆ 2019-01-0 ┆ 2019     ┆ 1       │
 │ Language   ┆            ┆ "cs.LG"]  ┆ 3         ┆   ┆ Tree-LSTM ┆ 1T00:15:4 ┆          ┆         │
 │ Processing ┆            ┆           ┆ figures,  ┆   ┆ with Tree ┆ 5Z        ┆          ┆         │
 │ …          ┆            ┆           ┆ Accepted  ┆   ┆ …        

In [61]:
df_after.filter(pl.col("version")>1)

abstract,arxiv_id,categories,comments,doi,num_categories,primary_cat,title,created,pub_year,version
str,str,list[str],str,str,i8,str,str,str,i16,i64
"""We develop a Bayesian bivariat…","""1901.00068""","[""stat.ME""]",null,null,1,"""stat.ME""","""A Bayesian Spatial Model for I…","""2019-01-01T00:32:45Z""",2019,4
"""We study the well-known minimu…","""1901.00491""","[""math.OC""]","""14 pages, 4 figures""",null,1,"""math.OC""","""Optimal control of the double …","""2019-01-01T01:34:22Z""",2019,2
"""The Dirac equation is used to …","""1901.00074""","[""physics.atom-ph""]","""9 pages, 4 figures, 2 tables""",null,1,"""physics.atom-ph""","""A Fully Relativistic Treatment…","""2019-01-01T01:47:19Z""",2019,4
"""If dark matter has a finite si…","""1901.00075""","[""hep-ph""]","""7 pages, 4 figures. Version ac…","""10.1103/PhysRevLett.124.041101""",1,"""hep-ph""","""Finite-size dark matter and it…","""2019-01-01T02:03:49Z""",2019,2
"""The activation function in neu…","""1901.05894""","[""cs.CV""]","""Accepted in 7th International …",null,1,"""cs.CV""","""LiSHT: Non-Parametric Linearly…","""2019-01-01T02:24:06Z""",2019,4
…,…,…,…,…,…,…,…,…,…,…
"""We perform a global QCD analys…","""2501.00665""","[""hep-ph"", ""hep-ex"", ""nucl-th""]","""40 pages, 11 figures, 14 table…",null,3,"""hep-ph""","""Strangeness in the proton from…","""2024-12-31T22:35:17Z""",2024,3
"""We prove that a rational pseud…","""2501.00667""","[""math.CO""]","""22 pages. Typos corrected and …",null,1,"""math.CO""","""Boundaries of pseudointegral p…","""2024-12-31T22:45:39Z""",2024,2
"""The profile of a neutron star …","""2501.00668""","[""nucl-th""]","""12 pages; 8 figures""",null,1,"""nucl-th""","""General features of the stella…","""2024-12-31T22:54:39Z""",2024,2


In [62]:
df_after.filter(pl.col("arxiv_id") == "1901.00068")

abstract,arxiv_id,categories,comments,doi,num_categories,primary_cat,title,created,pub_year,version
str,str,list[str],str,str,i8,str,str,str,i16,i64
"""We develop a Bayesian bivariat…","""1901.00068""","[""stat.ME""]",null,null,1,"""stat.ME""","""A Bayesian Spatial Model for I…","""2019-01-01T00:32:45Z""",2019,4


In [64]:
df = pl.concat([df_before, df_after.drop("version")])

In [65]:
df.group_by("pub_year").len().sort("pub_year").write_csv("data/arxiv_yearly_counts.csv")

In [66]:
df.group_by(
    pl.col("created").str.slice(0, 7).alias("created_yearmonth")
).len().sort("created_yearmonth").write_csv("data/arxiv_monthly_counts.csv")

In [67]:
df_unique = df.unique("arxiv_id")

duplicate_published_dois = df_unique.filter(
    pl.col("doi").is_not_null(),
    pl.col("doi")!="",
    pl.col("doi").is_duplicated()
).select("arxiv_id").to_series().unique().to_list()

print(df.height, df_unique.height) # number of duplicated id entries
print(len(duplicate_published_dois))
df.filter(
    pl.col("arxiv_id").is_null()
)

2634052 2634024
4025


abstract,arxiv_id,categories,comments,doi,num_categories,primary_cat,title,created,pub_year
str,str,list[str],str,str,i8,str,str,str,i16


In [68]:
df_cleaned = pl.concat(
    [
        df_unique.filter(~pl.col("arxiv_id").is_in(duplicate_published_dois)),
        df_unique.filter( pl.col("arxiv_id").is_in(duplicate_published_dois))
        .sort(
            "created", descending=False
        ).unique(
            "doi"
        )
    ]
)
df_cleaned


abstract,arxiv_id,categories,comments,doi,num_categories,primary_cat,title,created,pub_year
str,str,list[str],str,str,i8,str,str,str,i16
"""In this paper, we evaluate con…","""1509.07627""","[""cs.CV"", ""cs.AI"", ""cs.MM""]","""5 pages, 3 figures""",null,3,"""cs.CV""","""Feature Evaluation of Deep Con…","""2015-09-25""",2015
"""We investigate the impact of c…","""1703.05329""","[""hep-ph""]","""7 pages, 4 figures""","""10.1140/epjc/s10052-017-4908-4""",1,"""hep-ph""","""Spontaneous Charge Breaking in…","""2017-03-15""",2017
"""We use a continued fraction ap…","""1507.04538""","[""math.CO"", ""math-ph"", ""math.MP""]","""34 pages, 17 figures""","""10.4171/AIHPD/37""",3,"""math.CO""","""Comparing two statistical ense…","""2015-07-16""",2015
"""We characterize compact locall…","""math/0509038""","[""math.DG""]","""References updated""",null,1,"""math.DG""","""Locally conformal parallel $G_…","""2005-09-02""",2005
"""There is ever growing demand f…","""1809.02028""","[""cs.RO""]","""7 pages, 5 figures, Appearing …",null,1,"""cs.RO""","""Satellite Capture and Servicin…","""2018-09-06""",2018
…,…,…,…,…,…,…,…,…,…
"""grc4f is a Monte-Carlo package…","""hep-ph/9603394""","[""hep-ph""]","""30 pages, LaTeX, 5 pages posts…","""10.1016/S0010-4655(96)00126-9""",1,"""hep-ph""","""grc4f v1.0: a Four-fermion Eve…","""1996-03-23""",1996
"""We investigate the prospects f…","""hep-ph/0510272""","[""hep-ph""]","""14 pages, 1 figure""","""10.1103/PhysRevD.73.053003""",1,"""hep-ph""","""Neutrino Mass Matrices with Va…","""2005-10-20""",2005
"""We argue that in the quantum m…","""1305.1281""","[""hep-ph""]","""Prepared for the proceedings o…","""10.1007/978-4-431-55285-7""",1,"""hep-ph""","""Breaking so(4) symmetry withou…","""2013-05-06""",2013


In [69]:
(
    df_unique.filter( pl.col("arxiv_id").is_in(duplicate_published_dois))
        .sort(
            "created", descending=False
        ).unique(
            "doi"
        )
)

abstract,arxiv_id,categories,comments,doi,num_categories,primary_cat,title,created,pub_year
str,str,list[str],str,str,i8,str,str,str,i16
"""Using the instanton representa…","""0705.0927""","[""gr-qc""]","""16 pages. Accepted by the Inte…","""10.1142/S0218271812500599""",1,"""gr-qc""","""Instanton representation of Pl…","""2007-05-07""",2007
"""An efficient method is describ…","""cond-mat/0104182""","[""cond-mat.mtrl-sci""]","""2 pages, 0 figures""","""10.1088/0953-8984/14/11/302""",1,"""cond-mat.mtrl-sci""","""Efficient index handling of mu…","""2001-04-10""",2001
"""Magnetic properties of a singl…","""1611.08246""","[""cond-mat.mes-hall""]",null,"""10.1103/PhysRevB.96.125431""",1,"""cond-mat.mes-hall""","""Vacancy in graphene: insight o…","""2016-11-24""",2016
"""We consider the problem of qua…","""1510.02736""","[""quant-ph""]","""10 pages, 1 figure. This submi…","""10.1103/PhysRevA.93.052105""",1,"""quant-ph""","""Informational completeness in …","""2015-10-09""",2015
"""Oscillatory behavior of electr…","""0809.1213""","[""nucl-th"", ""astro-ph"", … ""nucl-ex""]","""extended but somewhat differen…","""10.1016/j.nuclphysa.2010.04.00…",4,"""nucl-th""","""Neutrino effects in two-body e…","""2008-09-07""",2008
…,…,…,…,…,…,…,…,…,…
"""Magneto-optical properties of …","""cond-mat/0204393""","[""cond-mat.str-el"", ""cond-mat.mes-hall""]","""18 pages, 1 figure, PDF file""","""10.1103/PhysRevE.67.011602""",2,"""cond-mat.str-el""","""Quantum Dimensional Zeeman Eff…","""2002-04-18""",2002
"""Empirical studies of the first…","""astro-ph/0001271""","[""astro-ph""]","""10 pages, Invited talk at ""X-r…","""10.1063/1.1434627""",1,"""astro-ph""","""Probing the Cosmic Dark Age in…","""2000-01-14""",2000
"""A hypergeometric type equation…","""quant-ph/0306012""","[""quant-ph""]","""10 pages, LaTeX2e in IOP style""","""10.2478/BF02476425""",1,"""quant-ph""","""Orthogonal polynomials defined…","""2003-06-02""",2003


In [33]:
# 2136612 - 2_134_777 + 1690
# 

In [70]:
(
    df_cleaned
    .with_columns(
        pl.col("doi").alias("curated_doi"),
        ("10.48550/arXiv." + pl.col("arxiv_id")).alias("arxiv_id"),
        pl.col("primary_cat").alias("primary_category"),
        pl.col("created").alias("published_date"),
    )
    .select([
        pl.col("arxiv_id").alias('doi'),
        pl.col("primary_category"),
        pl.col("curated_doi"),
        pl.col("published_date"),
    ]).write_ndjson(f"{data_path}/arxiv/arxiv_preprints.dedup_0129.jsonl")
)

In [113]:
df_cleaned.with_columns(
    pl.col("created").str.split("-").list[0].cast(pl.Int64())
).filter(pl.col("created")<2005)

abstract,arxiv_id,categories,comments,doi,num_categories,primary_cat,title,updated,created
str,str,list[str],str,str,i64,str,str,str,i64
"""Higher loop calculations in th…","""hep-ph/9611357""","[""hep-ph""]","""replaced by the published vers…","""10.1016/S0370-2693(96)01693-0""",1,"""hep-ph""","""On the position of a heavy Hig…","""1997-04-11""",1996
"""Elastic scattering of photons …","""hep-ph/9407245""","[""hep-ph""]","""8 pages, LATEX, BINP 94-65""","""10.1016/0375-9601(94)90242-9""",1,"""hep-ph""","""Relativistic Oscillator Model …",null,1994
"""A difficulty of studying dampe…","""astro-ph/9811274""","[""astro-ph""]","""4 pages LaTeX, including 1 fig…","""10.1086/311808""",1,"""astro-ph""","""The Closest Damped Lyman Alpha…",null,1998
"""We construct a compact symplec…","""dg-ga/9606007""","[""dg-ga"", ""math.DG""]","""LaTeX2e, 3 pages""",null,2,"""dg-ga""","""Example of a non-log-concave D…",null,1996
"""We investigate a double layer …","""cond-mat/9606135""","[""cond-mat""]","""7 pages, latex, 2 postscript f…","""10.1007/BF00768464""",1,"""cond-mat""","""Collective Spin Modes in Super…",null,1996
…,…,…,…,…,…,…,…,…,…
"""Different future long baseline…","""hep-ph/0210377""","[""hep-ph""]","""11 pages, 5 figures, Invited t…","""10.1016/S0920-5632(03)01318-5""",1,"""hep-ph""","""The Physics Potential of Futur…","""2002-11-20""",2002
"""Resonant active-to-active ($\n…","""astro-ph/0403529""","[""astro-ph""]","""Accepted for publication in th…","""10.1140/epjc/s2004-01816-3""",1,"""astro-ph""","""Strongest gravitational waves …",null,2004
"""We extend the formalism for th…","""astro-ph/0005390""","[""astro-ph""]","""15 pages + 2 figures, LaTeX wi…","""10.1086/308676""",1,"""astro-ph""","""Relativistic Corrections to th…",null,2000


In [ ]:
import json

curated_doi = df_cleaned.filter(
    pl.col("published").is_not_null(),
    pl.col("published")!=""
).select("doi", "published")
curated_doi_dict = dict(zip(curated_doi["doi"], curated_doi["published"]))

unique_doi = df_cleaned.select("doi").to_series().unique().to_list()

print(len(curated_doi_dict), len(unique_doi))

with open(f"{base_path}/curated_arxiv.json", "w") as f:
    json.dump(curated_doi_dict, f)
    
with open(f"{base_path}/arxiv_unique_doi.json", "w") as f:
    json.dump(unique_doi, f)

149892 262715
